In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [31]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std =[0.2470, 0.2435, 0.2616]),
])

In [32]:
train_dataset = datasets.ImageFolder(root='./datasets/cifar10/train', transform=transform)
test_dataset  = datasets.ImageFolder(root='./datasets/cifar10/test',  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(train_dataset.classes)
print(train_dataset.class_to_idx)

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
{'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}


In [33]:
print(len(train_dataset))
print(len(test_dataset))

50000
10000


In [34]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')


device = get_default_device()
device

device(type='cpu')

In [35]:
class CNN(nn.Module):  # nn.Module: PyTorch 신경망 모듈 base class 상속
    def __init__(self):
        super().__init__()  # nn.Module 생성자 호출
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),  # 입력 채널: 32, 앞 Conv2d Layer의 출력 채널 32를 그대로 받기 위함
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(  
            nn.Linear(64*7*7, 128),  # Conv2d(28x28) -> MaxPool2d(14x14) -> Conv2d(14x14) -> MaxPool2d(7x7) = 64x7x7
            nn.ReLU(),
            nn.Dropout(0.3),  # training 중 무작위로 30%의 뉴런 비활성화, 과적합 방지를 위한 정규화 기법
            nn.Linear(128, 10)
        )

    def forward(self, x):  # nn.Module forward() overriding
        x = self.conv(x)
        x = x.view(-1, 64*7*7)  # 3D tensor -> 1D tensor로 Flattening
        return self.fc(x)

model = CNN()

In [36]:
criterion = nn.CrossEntropyLoss()  # CrossEntropyLoss function
optimizer = optim.Adam(model.parameters(), lr=0.0007)  # Adam optimizer, learning rate: 0.0007

In [37]:
train_losses, test_losses         = [], []
train_accuracies, test_accuracies = [], []

In [40]:
epochs = 10
classes = train_dataset.classes

for epoch in range(epochs):

    # ── Train ──
    model.train()  # 모델 학습 진행
    running_loss, correct, total = 0, 0, 0

    for images, labels in enumerate(train_loader, 0):
        images, labels = labels
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()  # loss값을 통해 현재 기울기를 계산하여 파라미터 값을 .grad에 저장
        optimizer.step() # .grad 값을 기반으로 최적의 파라미터 계산 후 가중치 업데이트
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_losses.append(running_loss / len(train_loader))
    train_accuracies.append(100 * correct / total)

    # ── Test ──
    model.eval()  # 모델 추론 진행
    correct_test = 0
    total_test = 0
    test_loss = 0

    with torch.no_grad():
        for images, labels in test_loader:
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_loss = test_loss / len(test_loader)
    test_acc = 100 * correct_test / total_test

    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Test Loss: {test_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx